# SkylineGeolocation — Avenue A/B/C GPU Evaluation

Tests three avenues that require GPU:
- **Avenue A**: SAM 2 zero-shot sky segmentation (full-res 1080×720 masks)
- **Avenue B**: DINOv2 ViT-S/14 feature matching (2D vision-transformer cosine similarity)
- **Avenue C**: PnP RANSAC peak constellation (already tested locally — loaded from Drive)

**All checkpoints saved to `MyDrive/avenue_checkpoints/`. Re-run skips completed work.**

Required Drive files:
- `MyDrive/skyline_db.parquet`
- `MyDrive/street_view/` (images/, masks/, ground_truth.json, annotations.json)
- `MyDrive/sky_segmentation_unet_model.pth`

GPU: T4 (16GB) recommended. Avenue A+B take ~20 min total.

## 1. Setup

In [11]:
import os, shutil
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
BRANCH = 'main'

if (REPO / 'src').exists():
    print('Repo already at', REPO)
else:
    print('Cloning repo...')
    if REPO.is_file() or REPO.is_symlink():
        REPO.unlink()
    elif REPO.is_dir():
        shutil.rmtree(REPO)
    REPO.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', BRANCH,
         'https://github.com/pxrxp/SkylineGeolocation.git', str(REPO)],
        check=True, capture_output=True, text=True
    )
    print('Cloned. Branch:', BRANCH)
import sys
sys.path.insert(0, str(REPO))

Repo already at /content/SkylineGeolocation


In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


In [13]:
!pip install -q timm torchvision fastdtw pyarrow geopy
!apt-get install -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null
print('Deps ready')


Deps ready


In [14]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')
CKPT_DIR = DRIVE / 'avenue_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = CKPT_DIR / 'state.json'

links = [
    (DRIVE / 'skyline_db.parquet',
     REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'),
    (DRIVE / 'street_view',
     REPO / 'data/street_view'),
    (DRIVE / 'sky_segmentation_unet_model.pth',
     REPO / 'data/sky_segmentation_unet_model.pth'),
]

for src, dst in links:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')

skyline_db.parquet: already linked
street_view: already linked
sky_segmentation_unet_model.pth: already linked


In [6]:
db_path = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
if db_path.exists():
    size_mb = db_path.stat().st_size / 1024 / 1024
    with open(db_path, 'rb') as f:
        f.seek(0); hdr = f.read(4)
        f.seek(-8, 2); ftr = f.read()
    valid = hdr == b'PAR1' and ftr[:4] == b'PAR1'
    print(f'DB: {size_mb:.0f}MB, valid={valid}')
else:
    print('DB MISSING — cannot proceed')

DB: 463MB, valid=False


In [7]:
import json, time
from pathlib import Path

class CheckpointManager:
    """Per-sample checkpointing to Drive. Crash-safe: state written after each sample."""
    def __init__(self, state_path):
        self.path = Path(state_path)
        self.state = {}
        if self.path.exists():
            try:
                self.state = json.loads(self.path.read_text())
            except (json.JSONDecodeError, OSError):
                self.state = {}
    def is_done(self, avenue, sample_id):
        return self.state.get(avenue, {}).get(sample_id, {}).get('status') == 'ok'
    def mark_done(self, avenue, sample_id, **meta):
        self.state.setdefault(avenue, {})[sample_id] = {
            'status': 'ok', 'time': time.strftime('%Y-%m-%d %H:%M:%S'), **meta
        }
        self.save()
    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self.path.with_suffix('.tmp')
        tmp.write_text(json.dumps(self.state, indent=2, default=str))
        tmp.rename(self.path)  # atomic on same filesystem
    def summary(self):
        for ave in self.state:
            done = sum(1 for v in self.state[ave].values() if v.get('status') == 'ok')
            total = len(self.state[ave])
            print(f'  {ave}: {done}/{total} samples completed')

ck = CheckpointManager(STATE_PATH)
print('Checkpoint state:')
ck.summary() if ck.state else print('  Fresh start')

Checkpoint state:
  Fresh start


## 2. Avenue A: SAM 2 Zero-Shot Sky Segmentation

Replace U-Net 256×256 masks with SAM 2 full-res 1080×720 masks.
Point prompts at top of frame (sky) + bottom (terrain).

In [24]:
import json, os, sys, time
import numpy as np
from pathlib import Path
from PIL import Image

sys.path.insert(0, str(REPO))
from src.query_profile import extract_elevation_profile
from scripts.fixes_eval import Rx, mask_from_ann, DB_PATH, GT_FILE, ANNOT_FILE, CALIB_FILE

IMAGE_DIR = REPO / 'data/street_view/images'
MASK_DIR_A = CKPT_DIR / 'avenue_a_masks'
PROFILE_DIR_A = CKPT_DIR / 'avenue_a_profiles'
MASK_DIR_A.mkdir(parents=True, exist_ok=True)
PROFILE_DIR_A.mkdir(parents=True, exist_ok=True)

gt = json.load(open(REPO / 'data/street_view/ground_truth.json'))
ann = json.load(open(REPO / 'data/street_view/annotations.json'))['annotations']
calib = json.load(open(REPO / 'data/street_view/calibrated_ground_truth.json'))
sids = [s for s in ann if s in gt and (IMAGE_DIR / f'{s}.png').exists()][:17]
print(f'Samples: {len(sids)}')

need_a = [s for s in sids if not ck.is_done('avenue_a', s)]
print(f'Avenue A remaining: {len(need_a)}/{len(sids)}')

if need_a:
    import torch
    from hydra import compose, initialize_config_dir
    from hydra.core.global_hydra import GlobalHydra
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    # Download weights if missing
    ckpt_path = Path('sam2_hiera_large.pt')
    if not ckpt_path.exists():
        import urllib.request
        url = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'
        print(f'Downloading SAM2.1 Large weights...')
        urllib.request.urlretrieve(url, ckpt_path)
        print(f'Downloaded: {ckpt_path.stat().st_size / 1e6:.1f} MB')

    # Initialize Hydra with the configs/ subdirectory directly
    import sam2
    sam2_dir = str(Path(sam2.__file__).parent)
    configs_dir = os.path.join(sam2_dir, 'configs')

    GlobalHydra.instance().clear()
    initialize_config_dir(config_dir=configs_dir, version_base=None)
    sam2_model = build_sam2(
        'sam2.1/sam2.1_hiera_l',
        str(ckpt_path),
        device=str(device)
    )
    GlobalHydra.instance().clear()
    predictor = SAM2ImagePredictor(sam2_model)

    for si, sid in enumerate(need_a):
        t0 = time.time()
        g = gt[sid]
        img = np.array(Image.open(IMAGE_DIR / f'{sid}.png').convert('RGB'))
        H, W = img.shape[:2]

        predictor.set_image(img)

        # Point prompts: sky at top center, terrain at bottom center + corners
        pts = np.array([[W//2, 5], [W//2, H-5], [5, H-5], [W-5, H-5]])
        labels = np.array([1, 0, 0, 0])  # 1=sky, 0=terrain

        masks, scores, _ = predictor.predict(
            point_coords=pts,
            point_labels=labels,
            multimask_output=False
        )
        mask = masks[0]  # (H, W) - may be torch tensor or numpy bool

        # Convert to numpy bool if torch tensor
        if hasattr(mask, 'cpu'):
            mask = mask.cpu().numpy()
        mask = mask.astype(bool)

        # Invert: SAM2 mask sky=True -> need sky=255 for extract_elevation_profile convention
        mask_u8 = (~mask).astype(np.uint8) * 255  # sky=255, terrain=0

        # Save mask
        mask_path = MASK_DIR_A / f'{sid}.png'
        Image.fromarray(mask_u8).save(mask_path)

        # Extract profile using r_tilt (rotation matrix) from ground truth
        tilt = np.array(g['cam_R_tilt'])
        dp = float(calib.get(sid, {}).get('delta_pitch_deg', 0.0))
        # Apply pitch calibration: rotate by Rx(dp)
        if dp != 0.0:
            tilt = Rx(np.radians(dp)) @ tilt
        result = extract_elevation_profile(
            mask_u8, fov_y_deg=g['fov_y_deg'], r_tilt=tilt, bin_deg=0.5
        )
        profile_path = PROFILE_DIR_A / f'{sid}.npy'
        np.save(profile_path, result['profile'])

        ck.mark_done('avenue_a', sid)
        dt = time.time() - t0
        print(f'  [{si+1}/{len(need_a)}] {sid} score={scores[0]:.3f} {dt:.1f}s')

    del predictor, sam2_model
    torch.cuda.empty_cache()
    print('Avenue A complete.')
else:
    print('Avenue A already done.')


Samples: 17
Avenue A remaining: 17/17
Device: cuda
  [1/17] gtM1wHN3PY6ExdUpm_Kwxg score=0.992 1.2s
  [2/17] 7MjMNO0rcwbYlY0p0BbpZQ score=0.788 0.7s
  [3/17] 2d-jydcZQqi4LqAfXTTkGg score=0.984 0.7s
  [4/17] zkD6q2a9bU6jrCnCClr2VA score=0.989 0.7s
  [5/17] sLA2BviEGB1J0cWD9M25WA score=0.930 0.7s
  [6/17] CIHM0ogKEICAgIC6rYembA score=0.973 0.7s
  [7/17] CIHM0ogKEICAgIC6rYfGdw score=0.986 0.7s
  [8/17] Ofjf1ddyJ-EgeuduO3MIwA score=0.976 0.7s
  [9/17] MW24IuujFMDM2RjVTsVBFA score=0.472 0.7s
  [10/17] CIHM0ogKEICAgIC6rafUbQ score=0.988 0.7s
  [11/17] fxs4Vz2Y_jxf90ojEuemjQ score=0.833 0.8s
  [12/17] pc2xH30aP_n7tatAMCm0Qg score=0.960 0.8s
  [13/17] 4AQK-ijaR3ZiCtgy_PKW1g score=0.990 0.8s
  [14/17] ovVTNcM4FYYVEcEWX-Bmlg score=0.738 0.8s
  [15/17] rd3ozgCa3b9A94QaOFstLA score=0.935 0.8s
  [16/17] IJWw-IWxZJ1j0X5GhaTWPw score=0.000 0.8s
  [17/17] Huvxci3kfiIGJ9485Hxpkg score=0.111 0.7s
Avenue A complete.


## 3. Avenue B: DINOv2 Feature Matching

Encode query photos + DB horizon silhouette images with DINOv2 ViT-S/14.
Compare cosine similarity to rank candidate VPs.

In [26]:
import json, os, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from PIL import Image
from geopy.distance import geodesic
import torch
from torchvision import transforms as T

sys.path.insert(0, str(REPO))
from src.horizon_format import decode_horizon_uint8
from scripts.fixes_eval import Rx, DB_PATH, GT_FILE, ANNOT_FILE, CALIB_FILE

IMAGE_DIR = REPO / 'data/street_view/images'

# Load geometry
meta = pd.read_parquet(DB_PATH, columns=['lon', 'lat'])
vp_lon = meta['lon'].to_numpy()
vp_lat = meta['lat'].to_numpy()
del meta

gt = json.load(open(REPO / 'data/street_view/ground_truth.json'))
ann = json.load(open(REPO / 'data/street_view/annotations.json'))['annotations']
calib = json.load(open(REPO / 'data/street_view/calibrated_ground_truth.json'))
sids = [s for s in ann if s in gt and (IMAGE_DIR / f'{s}.png').exists()][:17]
need_b = [s for s in sids if not ck.is_done('avenue_b', s)]
print(f'Avenue B remaining: {len(need_b)}/{len(sids)}')

RESULTS_DIR_B = CKPT_DIR / 'avenue_b_results'
RESULTS_DIR_B.mkdir(parents=True, exist_ok=True)

if need_b:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True)
    dino_model = dino_model.to(device).eval()
    dino_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def horizon_to_silhouette(horizon, W=224, H=224):
        L = len(horizon)
        img = np.zeros((H, W), dtype=np.uint8)
        for c in range(W):
            col_az = (c / W) * 360.0
            h_idx = int((col_az % 360.0) / (360.0 / L)) % L
            elev = horizon[h_idx]
            row = int(H * (1.0 - (elev + 10.0) / 100.0))
            row = max(0, min(H - 1, row))
            img[row:, c] = 255
        return img

    def encode_batch(images):
        tensors = []
        for img in images:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            tensors.append(dino_transform(img))
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            feats = dino_model(batch)
        return feats.cpu().numpy()

    # Pre-encode DB horizon silhouettes in batches (stride to fit memory)
    DB_STRIDE = 12
    print(f'Encoding DB horizons (stride={DB_STRIDE})...', flush=True)
    t0 = time.time()

    # First pass: count total
    pf = pq.ParquetFile(DB_PATH)
    total_rows = sum(batch.num_rows for batch in pf.iter_batches(batch_size=1, columns=['raw_horizon_deg']))
    n_selected = total_rows // DB_STRIDE
    print(f'Total VPs: {total_rows}, selected: {n_selected}')

    # Encode in batches
    db_feats = []
    db_indices = []
    batch_imgs = []
    batch_idx = []
    BATCH_SIZE = 64
    row_count = 0

    for batch in pf.iter_batches(batch_size=4096, columns=['raw_horizon_deg']):
        for local_i in range(batch.num_rows):
            if row_count % DB_STRIDE != 0:
                row_count += 1
                continue
            horizon = decode_horizon_uint8(batch['raw_horizon_deg'][local_i].as_py())
            sil = horizon_to_silhouette(horizon)
            batch_imgs.append(Image.fromarray(sil))
            batch_idx.append(row_count)
            row_count += 1

            if len(batch_imgs) >= BATCH_SIZE:
                feats = encode_batch(batch_imgs)
                db_feats.append(feats)
                db_indices.extend(batch_idx)
                batch_imgs = []
                batch_idx = []

    if batch_imgs:
        feats = encode_batch(batch_imgs)
        db_feats.append(feats)
        db_indices.extend(batch_idx)

    db_feats = np.concatenate(db_feats, axis=0)
    db_indices = np.array(db_indices)
    print(f'Encoded {len(db_feats)} DB silhouettes in {time.time()-t0:.0f}s')

    # Now match each query
    for si, sid in enumerate(need_b):
        t0 = time.time()
        g = gt[sid]
        img = Image.open(IMAGE_DIR / f'{sid}.png').convert('RGB')
        query_feat = encode_batch([img])[0]  # (384,)
        query_feat = query_feat / np.linalg.norm(query_feat)

        # Cosine similarity
        db_norms = db_feats / (np.linalg.norm(db_feats, axis=1, keepdims=True) + 1e-8)
        sims = db_norms @ query_feat

        # Get top match
        top_idx = np.argmax(sims)
        top_vp = db_indices[top_idx]
        top_lon, top_lat = vp_lon[top_vp], vp_lat[top_vp]
        err_km = geodesic((g['true_lat'], g['true_lon']), (top_lat, top_lon)).km

        # True VP rank
        true_vp = int(g['closest_viewpoint_id'])
        true_sims = sims
        rank = int(np.sum(sims > sims[db_indices == true_vp][0])) + 1 if true_vp in db_indices else -1

        result = {
            'sid': sid,
            'top1_vp': int(top_vp),
            'top1_err_km': round(err_km, 2),
            'top1_corr': round(float(sims[top_idx]), 4),
            'true_rank': rank,
            'true_corr': round(float(sims[db_indices == true_vp][0]), 4) if true_vp in db_indices else -1,
            'profile_len': len(sims),
        }
        (RESULTS_DIR_B / f'{sid}.json').write_text(json.dumps(result, indent=2))
        ck.mark_done('avenue_b', sid)

        dt = time.time() - t0
        print(f'  [{si+1}/{len(need_b)}] {sid} err={err_km:.1f}km rank={rank} {dt:.1f}s')

    del dino_model, db_feats
    torch.cuda.empty_cache()
    print('Avenue B complete.')
else:
    print('Avenue B already done.')


Avenue B remaining: 17/17
Device: cuda


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Encoding DB horizons (stride=12)...
Total VPs: 1338650, selected: 111554
Encoded 111555 DB silhouettes in 762s
  [1/17] gtM1wHN3PY6ExdUpm_Kwxg err=18.3km rank=-1 0.2s
  [2/17] 7MjMNO0rcwbYlY0p0BbpZQ err=17.7km rank=-1 0.2s
  [3/17] 2d-jydcZQqi4LqAfXTTkGg err=21.5km rank=-1 0.2s
  [4/17] zkD6q2a9bU6jrCnCClr2VA err=13.7km rank=-1 0.2s
  [5/17] sLA2BviEGB1J0cWD9M25WA err=13.9km rank=-1 0.2s
  [6/17] CIHM0ogKEICAgIC6rYembA err=20.0km rank=-1 0.2s
  [7/17] CIHM0ogKEICAgIC6rYfGdw err=15.1km rank=-1 0.2s
  [8/17] Ofjf1ddyJ-EgeuduO3MIwA err=11.2km rank=-1 0.2s
  [9/17] MW24IuujFMDM2RjVTsVBFA err=17.1km rank=-1 0.2s
  [10/17] CIHM0ogKEICAgIC6rafUbQ err=23.2km rank=-1 0.2s
  [11/17] fxs4Vz2Y_jxf90ojEuemjQ err=16.0km rank=-1 0.2s
  [12/17] pc2xH30aP_n7tatAMCm0Qg err=12.2km rank=-1 0.2s
  [13/17] 4AQK-ijaR3ZiCtgy_PKW1g err=3.9km rank=-1 0.2s
  [14/17] ovVTNcM4FYYVEcEWX-Bmlg err=21.8km rank=-1 0.2s
  [15/17] rd3ozgCa3b9A94QaOFstLA err=9.9km rank=-1 0.2s
  [16/17] IJWw-IWxZJ1j0X5GhaTWPw err=4.5km ra

## 4. Aggregate Results

In [29]:
import json, os
import numpy as np
from pathlib import Path
from geopy.distance import geodesic
import sys

sys.path.insert(0, str(REPO))

print('=== Checkpoint Status ===')
ck.summary()

# Avenue A summary
print('\n=== Avenue A: SAM 2 Masks ===')
a_results = []
for f in sorted((PROFILE_DIR_A).glob('*.npy')):
    sid = f.stem
    prof = np.load(f)
    a_results.append({'sid': sid, 'profile_len': len(prof), 'profile_std': round(float(np.std(prof)), 2)})
if a_results:
    print(f'  {len(a_results)} profiles extracted')
    stds = [r['profile_std'] for r in a_results]
    print(f'  Profile std: med={np.median(stds):.2f}  min={np.min(stds):.2f}  max={np.max(stds):.2f}')
else:
    print('  No profiles yet')

# Avenue B summary
print('\n=== Avenue B: DINOv2 Feature Matching ===')
b_results = []
for f in sorted(RESULTS_DIR_B.glob('*.json')):
    b_results.append(json.loads(f.read_text()))
if b_results:
    errs = np.array([r['top1_err_km'] for r in b_results])
    ranks = np.array([r['true_rank'] for r in b_results if r['true_rank'] > 0])
    print(f'  Samples: {len(b_results)}')
    print(f'  Top-1 error: med={np.median(errs):.1f}km  '
          f'<1km={int(np.sum(errs < 1))}/{len(errs)}  '
          f'<5km={int(np.sum(errs < 5))}/{len(errs)}  '
          f'<10km={int(np.sum(errs < 10))}/{len(errs)}')
    if len(ranks) > 0:
        print(f'  True-VP rank: med={int(np.median(ranks)):d}  mean={np.mean(ranks):.0f}')
    print('\n  Per-sample:')
    for r in sorted(b_results, key=lambda x: x['top1_err_km']):
        print(f'    {r["sid"][:25]:25s} err={r["top1_err_km"]:6.1f}km  rank={r["true_rank"]:5d}')
else:
    print('  No results yet')

# Save combined summary
summary = {
    'avenue_a': a_results,
    'avenue_b': b_results,
}
(CKPT_DIR / 'combined_summary.json').write_text(json.dumps(summary, indent=2))
print(f'\nSummary saved to {CKPT_DIR / "combined_summary.json"}')

=== Checkpoint Status ===
  avenue_a: 17/17 samples completed
  avenue_b: 17/17 samples completed

=== Avenue A: SAM 2 Masks ===
  17 profiles extracted
  Profile std: med=8.76  min=3.06  max=18.54

=== Avenue B: DINOv2 Feature Matching ===
  Samples: 17
  Top-1 error: med=15.1km  <1km=0/17  <5km=2/17  <10km=4/17
  True-VP rank: med=43713  mean=43713

  Per-sample:
    4AQK-ijaR3ZiCtgy_PKW1g    err=   3.9km  rank=   -1
    IJWw-IWxZJ1j0X5GhaTWPw    err=   4.5km  rank=43713
    Huvxci3kfiIGJ9485Hxpkg    err=   9.3km  rank=   -1
    rd3ozgCa3b9A94QaOFstLA    err=   9.9km  rank=   -1
    Ofjf1ddyJ-EgeuduO3MIwA    err=  11.2km  rank=   -1
    pc2xH30aP_n7tatAMCm0Qg    err=  12.2km  rank=   -1
    zkD6q2a9bU6jrCnCClr2VA    err=  13.7km  rank=   -1
    sLA2BviEGB1J0cWD9M25WA    err=  13.9km  rank=   -1
    CIHM0ogKEICAgIC6rYfGdw    err=  15.1km  rank=   -1
    fxs4Vz2Y_jxf90ojEuemjQ    err=  16.0km  rank=   -1
    MW24IuujFMDM2RjVTsVBFA    err=  17.1km  rank=   -1
    7MjMNO0rcwbYlY0p0BbpZQ 

In [2]:
import json, time
import numpy as np
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
from geopy.distance import geodesic
import pyarrow.parquet as pq

DRIVE = Path('/content/drive/MyDrive')
CKPT = DRIVE / 'avenue_checkpoints'
DB_PATH = DRIVE / 'skyline_db.parquet'
GT_FILE = DRIVE / 'street_view' / 'ground_truth.json'
ANNOT_FILE = DRIVE / 'street_view' / 'annotations.json'
CLEAN_PROF = CKPT / 'avenue_a_profiles_clean'

gt = json.load(open(GT_FILE))
ann = json.load(open(ANNOT_FILE))['annotations']
sids = sorted([s for s in ann if s in gt and (CLEAN_PROF / f'{s}.npy').exists()])
print(f'Samples: {len(sids)}')

meta = pq.read_table(DB_PATH, columns=['lat', 'lon'])
vp_lats = meta.column('lat').to_numpy()
vp_lons = meta.column('lon').to_numpy()
N, M = len(vp_lats), 720
print(f'DB: {N} VPs')

def bandpass(arr, s1=2.0, s2=8.0):
    return gaussian_filter1d(arr, s1, axis=-1) - gaussian_filter1d(arr, s2, axis=-1)

qf = {}
for sid in sids:
    q = np.load(CLEAN_PROF / f'{sid}.npy')
    qp = np.zeros(M); qp[:len(q)] = q
    qb = bandpass(qp.reshape(1, -1))[0]
    qf[sid] = (qb.mean(), qb.std(), np.fft.rfft(qb), len(q))

print('Matching full DB (no priors)...')
t0 = time.time()
sc = {sid: np.full(N, -999.0) for sid in sids}
pf = pq.ParquetFile(DB_PATH)
nb = (N + 4095) // 4096

for bi, batch in enumerate(pf.iter_batches(batch_size=4096, columns=['raw_horizon_deg'])):
    bsz = batch.num_rows
    b0 = bi * 4096
    raw = np.array(batch.column(0).to_pylist(), dtype=np.float64) * (90.0/255.0)
    dbp = bandpass(raw)
    cm = dbp.mean(axis=1, keepdims=True)
    cs = dbp.std(axis=1, keepdims=True) + 1e-12
    cf = np.fft.rfft(dbp, axis=1)
    for sid in sids:
        qm, qs, qft, qlen = qf[sid]
        corr = np.fft.irfft(qft.conj() * cf, n=M, axis=1)
        sc[sid][b0:b0+bsz] = (corr.mean(axis=1) - qm * cm.ravel()) / (qs * cs.ravel())
    if bi % 40 == 0:
        print(f'  {bi}/{nb} ({time.time()-t0:.0f}s)', flush=True)

t_honest = time.time() - t0
print(f'Done in {t_honest:.0f}s')

results_h = []
for sid in sids:
    g = gt[sid]
    s = sc[sid]
    bv = int(np.argmax(s))
    tv = int(g['closest_viewpoint_id'])
    ekm = geodesic((g['true_lat'],g['true_lon']),(vp_lats[bv],vp_lons[bv])).km
    ts = s[tv]
    rank = int(np.sum(s > ts)) + 1
    results_h.append({'sid':sid,'err_km':round(ekm,2),'rank':rank})

errs = np.array([r['err_km'] for r in results_h])
ranks = np.array([r['rank'] for r in results_h])
print(f'\n=== HONEST FULL-DB SCAN ({N} VPs, no priors) ===')
print(f'err: med={np.median(errs):.1f}km <1km={int(np.sum(errs<1))}/{len(errs)} <5km={int(np.sum(errs<5))}/{len(errs)} <10km={int(np.sum(errs<10))}/{len(errs)}')
print(f'rank: med={int(np.median(ranks))} mean={np.mean(ranks):.0f}')
for r in sorted(results_h, key=lambda x: x['err_km']):
    print(f'  {r["sid"][:25]:25s} err={r["err_km"]:7.1f}km rank={r["rank"]:6d}')

# === MOBILE PRIORS ===
print(f'\n=== MOBILE PRIORS: 5km cell + compass +20 deg + pitch 2 deg ===')
results_p = []
for sid in sids:
    g = gt[sid]
    tv_lat, tv_lon = g['true_lat'], g['true_lon']
    true_vp = int(g['closest_viewpoint_id'])
    dlat = vp_lats - tv_lat
    dlon = vp_lons - tv_lon
    approx_km = np.sqrt((dlat*111.32)**2 + (dlon*111.32*np.cos(np.radians(tv_lat)))**2)
    within = approx_km <= 5.0
    sp = sc[sid].copy(); sp[~within] = -999.0
    bv = int(np.argmax(sp))
    ekm = geodesic((tv_lat,tv_lon),(vp_lats[bv],vp_lons[bv])).km
    ts = sp[true_vp]
    rank = int(np.sum(sp > ts)) + 1
    results_p.append({'sid':sid,'err_km':round(ekm,2),'rank':rank,'n_gate':int(within.sum())})

errs_p = np.array([r['err_km'] for r in results_p])
ranks_p = np.array([r['rank'] for r in results_p])
print(f'5km prior: med={np.median(errs_p):.1f}km <1km={int(np.sum(errs_p<1))}/{len(errs_p)} <5km={int(np.sum(errs_p<5))}/{len(errs_p)} rank_med={int(np.median(ranks_p))}')
for r in sorted(results_p, key=lambda x: x['err_km']):
    print(f'  {r["sid"][:25]:25s} err={r["err_km"]:7.1f}km rank={r["rank"]:6d} VPs={r["n_gate"]}')

out = CKPT / 'sam2_clean_bp_results.json'
out.write_text(json.dumps({'honest':{'med':round(float(np.median(errs)),1),'lt1':int(np.sum(errs<1)),'lt5':int(np.sum(errs<5)),'lt10':int(np.sum(errs<10)),'rank_med':int(np.median(ranks)),'n_vp':N,'time_s':round(t_honest),'details':results_h},'prior5km':{'med':round(float(np.median(errs_p)),1),'lt1':int(np.sum(errs_p<1)),'lt5':int(np.sum(errs_p<5)),'rank_med':int(np.median(ranks_p)),'details':results_p}},indent=2))
print(f'\nSaved: {out}')

Samples: 17
DB: 1338650 VPs
Matching full DB (no priors)...
  0/327 (3s)
  40/327 (89s)
  80/327 (175s)
  120/327 (260s)
  160/327 (345s)
  200/327 (430s)
  240/327 (514s)
  280/327 (598s)
  320/327 (681s)
Done in 694s

=== HONEST FULL-DB SCAN (1338650 VPs, no priors) ===
err: med=15.9km <1km=0/17 <5km=1/17 <10km=5/17
rank: med=593297 mean=753994
  sLA2BviEGB1J0cWD9M25WA    err=    3.1km rank=1299200
  CIHM0ogKEICAgIC6rYembA    err=    6.6km rank=573104
  7MjMNO0rcwbYlY0p0BbpZQ    err=    6.8km rank=508861
  rd3ozgCa3b9A94QaOFstLA    err=    7.6km rank=1156873
  IJWw-IWxZJ1j0X5GhaTWPw    err=    7.9km rank=1318169
  pc2xH30aP_n7tatAMCm0Qg    err=   10.4km rank=281449
  4AQK-ijaR3ZiCtgy_PKW1g    err=   12.1km rank=276741
  MW24IuujFMDM2RjVTsVBFA    err=   14.2km rank=430181
  fxs4Vz2Y_jxf90ojEuemjQ    err=   15.9km rank=855542
  2d-jydcZQqi4LqAfXTTkGg    err=   15.9km rank=144021
  ovVTNcM4FYYVEcEWX-Bmlg    err=   17.6km rank=1304436
  CIHM0ogKEICAgIC6rYfGdw    err=   23.0km rank=561789

In [5]:
import json
from pathlib import Path
r = Path('/content/drive/MyDrive/avenue_checkpoints/sam2_clean_bp_results.json')
d = json.loads(r.read_text())
print('Keys:', list(d.keys()))
for k, v in d.items():
    if isinstance(v, dict):
        print(f"\n{k}: med={v.get('med','?')}km lt1={v.get('lt1','?')} lt5={v.get('lt5','?')} rank={v.get('rank_med','?')}")


Keys: ['honest', 'prior5km']

honest: med=15.9km lt1=0 lt5=1 rank=593297

prior5km: med=2.9km lt1=0 lt5=17 rank=39175


In [10]:
import json, numpy as np
from pathlib import Path
from geopy.distance import geodesic

CKPT = Path('/content/drive/MyDrive/avenue_checkpoints')
GT_FILE = Path('/content/drive/MyDrive/street_view/ground_truth.json')
gt = json.load(open(GT_FILE))

sc = np.load(CKPT / 'sam2_scores.npy')
db_par = np.load(CKPT / 'sam2_db_parallax.npy')
vp_lats = np.load(CKPT / 'sam2_vp_lats.npy')
vp_lons = np.load(CKPT / 'sam2_vp_lons.npy')

ann = json.load(open(Path('/content/drive/MyDrive/street_view/annotations.json')))['annotations']
CLEAN_PROF = CKPT / 'avenue_a_profiles_clean'
sids = sorted([s for s in ann if s in gt and (CLEAN_PROF / f'{s}.npy').exists()])

tv_lats = np.array([gt[s]['true_lat'] for s in sids])
tv_lons = np.array([gt[s]['true_lon'] for s in sids])
dlat = vp_lats[:,None] - tv_lats[None,:]
dlon = vp_lons[:,None] - tv_lons[None,:]
lat_ref = np.radians(tv_lats)[None,:]
approx_km = np.sqrt((dlat*111.32)**2 + (dlon*111.32*np.cos(lat_ref))**2)

def parallax(h, nn=8, nf=8):
    s = np.sort(h)[::-1]
    dn = np.std(np.diff(s[:nn]))
    df = np.std(np.diff(s[-nf:]))
    return dn / (df + 1e-12)

def row(name, ea, ra):
    return f'{name:<30s} {np.median(ea):5.1f}k {int(np.sum(ea<1)):>2d}/{len(ea)} {int(np.sum(ea<5)):>2d}/{len(ea)} {int(np.sum(ea<10)):>3d}/{len(ea)} {int(np.median(ra)):>8d}'

print(f'{"Method":<30s} {"Med":>6s} {"<1km":>5s} {"<5km":>5s} {"<10km":>6s} {"Rank":>8s}')
print('-'*65)

# Honest
h_err, h_rank = [], []
for qi, sid in enumerate(sids):
    s = sc[qi]; bv = int(np.argmax(s)); tv = int(gt[sid]['closest_viewpoint_id'])
    ekm = geodesic((gt[sid]['true_lat'],gt[sid]['true_lon']),(vp_lats[bv],vp_lons[bv])).km
    rank = int(np.sum(s > s[tv])) + 1
    h_err.append(ekm); h_rank.append(rank)
he, hr = np.array(h_err), np.array(h_rank)
print(row('Honest (1.34M VPs)', he, hr))

for rad in [5.0, 2.0, 1.0]:
    within = approx_km <= rad
    re, rr = [], []
    for qi, sid in enumerate(sids):
        sp = sc[qi].copy(); sp[~within[:,qi]] = -999.0
        bv = int(np.argmax(sp)); tv = int(gt[sid]['closest_viewpoint_id'])
        ekm = geodesic((gt[sid]['true_lat'],gt[sid]['true_lon']),(vp_lats[bv],vp_lons[bv])).km
        rank = int(np.sum(sp > sp[tv])) + 1
        re.append(ekm); rr.append(rank)
    re, rr = np.array(re), np.array(rr)
    print(row(f'{rad:.0f}km prior', re, rr))
    for r in sorted(zip(sids, re, rr), key=lambda x: x[1]):
        print(f'  {r[0][:25]:25s} err={r[1]:7.1f}km rank={r[2]:6d}')

# Parallax on 2km pool
within_2 = approx_km <= 2.0
pr_e, pr_r = [], []
for qi, sid in enumerate(sids):
    pool = np.where(within_2[:,qi])[0]
    if len(pool) == 0: pool = np.argsort(approx_km[:,qi])[:200]
    q = np.load(CLEAN_PROF / f'{sid}.npy')
    q_par = parallax(q)
    sp = sc[qi][pool]
    top20 = pool[np.argsort(sp)[-min(20, len(sp)):]]
    par_dist = np.abs(db_par[top20] - q_par)
    winner = top20[np.argmin(par_dist)]
    tv = int(gt[sid]['closest_viewpoint_id'])
    ekm = geodesic((gt[sid]['true_lat'],gt[sid]['true_lon']),(vp_lats[winner],vp_lons[winner])).km
    sp2 = sc[qi].copy(); sp2[~within_2[:,qi]] = -999.0
    rank = int(np.sum(sp2 > sp2[tv])) + 1
    pr_e.append(ekm); pr_r.append(rank)
    print(f'  {sid}: err={ekm:.1f}km rank={rank} q_par={q_par:.3f} db_par={db_par[winner]:.3f}')
pre, prr = np.array(pr_e), np.array(pr_r)
print(row('2km + Parallax', pre, prr))

out = CKPT / 'sam2_priors_results.json'
out.write_text(json.dumps({
    'honest': {'med': round(float(np.median(he)),1), 'lt1': int(np.sum(he<1)), 'lt5': int(np.sum(he<5)), 'lt10': int(np.sum(he<10)), 'rank_med': int(np.median(hr))},
    '5km': {'med': round(float(np.median(re)),1)},
    'parallax_2km': {'med': round(float(np.median(pre)),1), 'lt1': int(np.sum(pre<1)), 'lt5': int(np.sum(pre<5))}
}, indent=2))
print(f'\nSaved: {out}')


Method                            Med  <1km  <5km  <10km     Rank
-----------------------------------------------------------------
Honest (1.34M VPs)              16.0k  0/17  1/17   4/17   830289
5km prior                        3.3k  0/17 17/17  17/17    53292
  4AQK-ijaR3ZiCtgy_PKW1g    err=    1.4km rank=  9850
  Ofjf1ddyJ-EgeuduO3MIwA    err=    1.6km rank= 70332
  pc2xH30aP_n7tatAMCm0Qg    err=    2.3km rank= 39572
  sLA2BviEGB1J0cWD9M25WA    err=    2.8km rank= 64657
  CIHM0ogKEICAgIC6rYembA    err=    2.9km rank= 68586
  gtM1wHN3PY6ExdUpm_Kwxg    err=    2.9km rank= 29102
  CIHM0ogKEICAgIC6rafUbQ    err=    3.1km rank=  6564
  ovVTNcM4FYYVEcEWX-Bmlg    err=    3.3km rank= 53707
  MW24IuujFMDM2RjVTsVBFA    err=    3.3km rank= 48615
  2d-jydcZQqi4LqAfXTTkGg    err=    3.5km rank= 53292
  zkD6q2a9bU6jrCnCClr2VA    err=    3.7km rank=  9381
  rd3ozgCa3b9A94QaOFstLA    err=    3.7km rank= 45020
  fxs4Vz2Y_jxf90ojEuemjQ    err=    3.7km rank= 56180
  CIHM0ogKEICAgIC6rYfGdw    err=  